# Figure 4 — public-source TRACERx multiregion analysis

This notebook reconstructs every Figure 4 plotting table directly from the
public TRACERx Renal files acquired by `00_Paper_Data_Download.ipynb`.

It contains no archived effect sizes, P values, confidence intervals, FDRs, or
variance estimates, and it performs no comparisons with expected results.
Checks are limited to source integrity, cohort structure, model validity, and
the processed-file contract used by the unchanged plotting notebook.


In [1]:
from pathlib import Path
import hashlib
import os
import re
import warnings

import numpy as np
import pandas as pd
import pyreadr
import statsmodels.api as sm
from scipy.stats import pearsonr, spearmanr
from statsmodels.stats.multitest import multipletests

PROJECT_ROOT = Path(
    os.environ.get("MITOXY_PROJECT_ROOT", "/scratch/ar58064/mitoxy")
).resolve()
MANIFEST_PATH = (
    PROJECT_ROOT / "data" / "provenance" /
    "paper_public_source_manifest.csv"
)
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "figure_4_clean"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GENES = ["BAK1", "BAX", "BID", "SLC7A11"]
manifest = pd.read_csv(MANIFEST_PATH)


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def source_path(dataset, role):
    match = manifest.loc[
        manifest["dataset"].eq(dataset)
        & manifest["role"].eq(role)
    ]
    if len(match) != 1:
        raise RuntimeError(
            f"Expected one source for ({dataset!r}, {role!r}); "
            f"found {len(match)}"
        )
    row = match.iloc[0]
    path = PROJECT_ROOT / str(row["relative_path"])
    if not path.is_file():
        raise FileNotFoundError(path)
    if path.stat().st_size != int(row["size_bytes"]):
        raise RuntimeError(f"Source-size mismatch: {path}")
    if sha256_file(path) != str(row["sha256"]):
        raise RuntimeError(f"Source SHA-256 mismatch: {path}")
    return path


RAW = {
    "tpm": source_path("TRACERx Renal", "regional tumor TPM"),
    "annotation": source_path("TRACERx Renal", "regional annotation"),
    "clinical": source_path("TRACERx Renal", "clinical annotation"),
}

display(pd.DataFrame({"analysis_key": RAW.keys(), "path": RAW.values()}))
print("Public-source integrity audit: PASS")


,analysis_key,path
0,tpm,/scratch/ar58064/mitoxy/data/raw/tracerx_renal...
1,annotation,/scratch/ar58064/mitoxy/data/raw/tracerx_renal...
2,clinical,/scratch/ar58064/mitoxy/data/raw/tracerx_renal...


Public-source integrity audit: PASS


In [2]:
def zscore(values):
    values = pd.Series(values, index=getattr(values, "index", None), dtype=float)
    standard_deviation = values.std(ddof=0)
    if not np.isfinite(standard_deviation) or standard_deviation == 0:
        raise ValueError("Cannot standardize a constant or invalid vector")
    return (values - values.mean()) / standard_deviation


def normalized(text):
    return re.sub(r"[^a-z0-9]+", " ", str(text).lower()).strip()


def find_column(columns, include, exclude=(), required=True):
    include = [normalized(term) for term in include]
    exclude = [normalized(term) for term in exclude]
    candidates = []
    for column in columns:
        name = normalized(column)
        if all(term in name for term in include) and not any(
            term in name for term in exclude
        ):
            candidates.append(column)
    if not candidates:
        if required:
            raise KeyError(
                f"No column matched include={include}, exclude={exclude}. "
                f"Available columns: {list(columns)}"
            )
        return None
    return sorted(candidates, key=lambda value: (len(str(value)), str(value)))[0]


def parse_grade(value):
    if pd.isna(value):
        return np.nan
    match = re.search(r"(?:G(?:RADE)?\s*)?([1-4])", str(value), flags=re.I)
    return float(match.group(1)) if match else np.nan


def parse_stage(value):
    if pd.isna(value):
        return np.nan
    text = str(value).upper().replace("STAGE", " ")
    match = re.search(r"\b(IV|III|II|I)\b", text)
    if match:
        return {"I": 1.0, "II": 2.0, "III": 3.0, "IV": 4.0}[
            match.group(1)
        ]
    match = re.search(r"\b([1-4])\b", text)
    return float(match.group(1)) if match else np.nan


def fit_hc3(data, outcome, predictors, focal):
    fields = [outcome, *predictors]
    complete = data[fields].replace([np.inf, -np.inf], np.nan).dropna()
    design = sm.add_constant(complete[predictors], has_constant="add")
    model = sm.OLS(
        complete[outcome].astype(float), design.astype(float)
    ).fit(cov_type="HC3", use_t=False)
    interval = model.conf_int().loc[focal]
    return {
        "n": int(model.nobs),
        "beta": float(model.params[focal]),
        "se_HC3": float(model.bse[focal]),
        "p_value": float(model.pvalues[focal]),
        "ci_low": float(interval.iloc[0]),
        "ci_high": float(interval.iloc[1]),
    }


def partial_spearman(data, x, y, covariates):
    fields = [x, y, *covariates]
    complete = data[fields].replace([np.inf, -np.inf], np.nan).dropna()
    ranked = complete.rank(method="average")
    design = sm.add_constant(ranked[covariates], has_constant="add")
    x_residual = sm.OLS(ranked[x], design).fit().resid
    y_residual = sm.OLS(ranked[y], design).fit().resid
    rho, p_value = pearsonr(x_residual, y_residual)
    return int(len(complete)), float(rho), float(p_value)


def dispersion_summary(values):
    values = np.asarray(values, dtype=float)
    differences = np.abs(values[:, None] - values[None, :])
    pairwise = differences[np.triu_indices(len(values), k=1)]
    return pd.Series({
        "n_regions": len(values),
        "Range": np.ptp(values),
        "SD": np.std(values, ddof=1),
        "MAD": np.median(np.abs(values - np.median(values))),
        "Mean pairwise |difference|": np.mean(pairwise),
        "Median pairwise |difference|": np.median(pairwise),
    })


## Public TRACERx regional and patient-level reconstruction


In [3]:
expression = pyreadr.read_r(str(RAW["tpm"]))[None]
expression.index = expression.index.astype(str).str.strip().str.upper()
expression.columns = expression.columns.astype(str).str.strip()
missing_genes = sorted(set(GENES) - set(expression.index))
if missing_genes:
    raise KeyError(f"TRACERx expression is missing {missing_genes}")

annotation = pd.read_csv(RAW["annotation"], sep="\t", dtype=str)
annotation.columns = [str(column).strip() for column in annotation.columns]

sample_column = find_column(annotation.columns, ["sample"])
patient_column = find_column(annotation.columns, ["patient"])
type_column = find_column(annotation.columns, ["type", "collapsed"])
condition_column = find_column(annotation.columns, ["condition"])
purity_column = find_column(annotation.columns, ["purity"])
wgi_column = find_column(annotation.columns, ["wgi"], required=False)
if wgi_column is None:
    wgi_column = find_column(annotation.columns, ["weighted", "genome"])
ith_column = find_column(annotation.columns, ["ith"], required=False)
if ith_column is None:
    ith_column = find_column(annotation.columns, ["intratum", "heterogeneity"])

annotation[sample_column] = annotation[sample_column].astype(str).str.strip()
annotation[patient_column] = annotation[patient_column].astype(str).str.strip()
primary_annotation = annotation.loc[
    annotation[type_column].astype(str).str.strip().str.upper().eq("PRIMARY")
].copy()

primary_samples = [
    sample for sample in primary_annotation[sample_column]
    if sample in expression.columns
]
if not primary_samples:
    raise RuntimeError("No PRIMARY annotation rows overlap expression samples")

# Figure 4 follows the TRACERx evolutionary analysis and standardizes the
# public TPM values directly across primary-tumor regions.
region_expression = (
    expression.loc[GENES, primary_samples]
    .T.apply(pd.to_numeric, errors="coerce")
)
if region_expression.isna().any().any():
    raise ValueError("Missing target-gene TPM values in primary regions")
region_gene_z = region_expression.apply(zscore, axis=0)
region_gene_z.columns = [f"z_{gene}" for gene in GENES]

regions = region_gene_z.copy()
regions["mitoxy"] = region_gene_z.mean(axis=1)
regions["sample_id"] = regions.index.astype(str)
regions = regions.merge(
    primary_annotation[[sample_column, patient_column]].drop_duplicates(),
    left_on="sample_id", right_on=sample_column,
    how="left", validate="one_to_one",
).rename(columns={patient_column: "patient"})

clinical = pd.read_csv(RAW["clinical"], sep="\t", header=1, dtype=str)
clinical.columns = [str(column).strip() for column in clinical.columns]
clinical_patient = find_column(clinical.columns, ["subject"])
clinical_grade = find_column(clinical.columns, ["fuhrman", "grade"])
clinical_stage = find_column(clinical.columns, ["overall", "stage"])
clinical_age = find_column(clinical.columns, ["age"])

clinical_table = pd.DataFrame({
    "patient": clinical[clinical_patient].astype(str).str.strip(),
    "grade": clinical[clinical_grade].map(parse_grade),
    "stage": clinical[clinical_stage].map(parse_stage),
    "age": pd.to_numeric(clinical[clinical_age], errors="coerce"),
}).drop_duplicates("patient")

patient_annotation = primary_annotation[
    [patient_column, condition_column]
].groupby(patient_column, as_index=False).first().rename(columns={
    patient_column: "patient", condition_column: "condition",
})

covariate_rows = primary_annotation[
    [patient_column, purity_column, wgi_column, ith_column]
].rename(columns={
    patient_column: "patient", purity_column: "purity",
    wgi_column: "wgi", ith_column: "ith",
})
for column in ["purity", "wgi", "ith"]:
    covariate_rows[column] = pd.to_numeric(
        covariate_rows[column], errors="coerce"
    )
covariates = covariate_rows.groupby("patient", as_index=False)[
    ["purity", "wgi", "ith"]
].mean()

patients = regions.groupby("patient", as_index=False).agg(
    mitoxy_raw=("mitoxy", "mean"),
    n_regions=("mitoxy", "size"),
)
patients["mitoxy"] = zscore(patients["mitoxy_raw"])
patients = (
    patients
    .merge(clinical_table, on="patient", how="left", validate="one_to_one")
    .merge(patient_annotation, on="patient", how="left", validate="one_to_one")
    .merge(covariates, on="patient", how="left", validate="one_to_one")
)
patients["punctuated"] = patients["condition"].astype(str).str.casefold().eq(
    "punctuated"
).astype(float)

print("Primary regions:", len(regions))
print("Patients represented:", patients["patient"].nunique())
display(patients[[
    "grade", "stage", "age", "purity", "wgi", "ith", "condition"
]].notna().sum().rename("n_complete").to_frame())


Primary regions: 191
Patients represented: 75


,n_complete
grade,73
stage,73
age,73
purity,75
wgi,75
ith,75
condition,75


## Panel B — observed regional variation


In [4]:
validated = patients.dropna(
    subset=["grade", "stage", "age", "purity", "wgi", "ith", "condition"]
).copy()
validated_ids = set(validated["patient"])

panel_regions = regions.loc[
    regions["patient"].isin(validated_ids) & regions["mitoxy"].notna()
].copy()
patient_summary = panel_regions.groupby("patient", as_index=False).agg(
    patient_mean=("mitoxy", "mean"),
    patient_min=("mitoxy", "min"),
    patient_max=("mitoxy", "max"),
    n_regions=("mitoxy", "size"),
)
multiregion = patient_summary.loc[
    patient_summary["n_regions"].ge(2)
].sort_values("patient_mean").reset_index(drop=True)
multiregion["patient_order"] = np.arange(1, len(multiregion) + 1)
multiregion["patient_label"] = [
    f"P{index:02d}" for index in multiregion["patient_order"]
]

order_map = multiregion.set_index("patient")["patient_order"]
label_map = multiregion.set_index("patient")["patient_label"]
panel_b = panel_regions.loc[
    panel_regions["patient"].isin(multiregion["patient"])
].copy()
panel_b["patient_order"] = panel_b["patient"].map(order_map)
panel_b["patient_label"] = panel_b["patient"].map(label_map)
panel_b = panel_b.merge(
    multiregion[["patient", "patient_mean", "n_regions"]],
    on="patient", how="left", validate="many_to_one",
)

if len(validated) != 73 or len(panel_regions) != 186:
    warnings.warn(
        "The public complete-case cohort differs from the manuscript cohort: "
        f"{len(validated)} patients and {len(panel_regions)} regions."
    )

panel_b.to_csv(OUTPUT_DIR / "panel_b_region_scores.csv", index=False)
multiregion.to_csv(OUTPUT_DIR / "panel_b_patient_summary.csv", index=False)
print(
    f"Panel B: {len(panel_b)} regions across "
    f"{panel_b['patient'].nunique()} multiregion patients"
)
display(multiregion.head())


Panel B: 171 regions across 58 multiregion patients


,patient,patient_mean,patient_min,patient_max,n_regions,patient_order,patient_label
0,K145,-0.937808,-0.999517,-0.876100,2,1,P01
1,K520,-0.830473,-0.870850,-0.790096,2,2,P02
2,K162,-0.770272,-0.869385,-0.671159,2,3,P03
3,K065,-0.746674,-0.849338,-0.644010,2,4,P04
4,K059,-0.622969,-0.896641,-0.301834,3,5,P05


## Panel C — patient and within-tumor variance partition


In [5]:
variance_data = regions[["patient", "mitoxy"]].dropna().copy()
design = np.ones((len(variance_data), 1))
mixed_model = sm.MixedLM(
    variance_data["mitoxy"].astype(float),
    design,
    groups=variance_data["patient"],
)
try:
    mixed_result = mixed_model.fit(reml=True, method="lbfgs", disp=False)
except Exception:
    mixed_result = mixed_model.fit(reml=True, method="powell", disp=False)

between_variance = float(np.asarray(mixed_result.cov_re)[0, 0])
within_variance = float(mixed_result.scale)
total_variance = between_variance + within_variance
if not np.isfinite(total_variance) or total_variance <= 0:
    raise RuntimeError("Invalid variance components from the mixed model")

panel_c = pd.DataFrame({
    "component": ["Between patients", "Within tumor"],
    "variance_fraction": [
        between_variance / total_variance,
        within_variance / total_variance,
    ],
})
panel_c["variance_percent"] = 100 * panel_c["variance_fraction"]
panel_c.to_csv(OUTPUT_DIR / "panel_c_variance_partition.csv", index=False)

print(mixed_result.summary())
display(panel_c)


         Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: mitoxy   
No. Observations: 191     Method:             REML     
No. Groups:       75      Scale:              0.1934   
Min. group size:  1       Log-Likelihood:     -168.0900
Max. group size:  12      Converged:          Yes      
Mean group size:  2.5                                  
-------------------------------------------------------
             Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-------------------------------------------------------
const        -0.019    0.068 -0.277 0.782 -0.153  0.115
Group Var     0.258    0.169                           



,component,variance_fraction,variance_percent
0,Between patients,0.571744,57.17443
1,Within tumor,0.428256,42.82557


## Panel D — adjusted punctuated-evolution contrast


In [6]:
model_specs = [
    ("Clinical", ["punctuated", "purity", "age", "grade", "stage"]),
    ("Clinical + WGI", [
        "punctuated", "purity", "age", "grade", "stage", "wgi",
    ]),
    ("Clinical + ITH", [
        "punctuated", "purity", "age", "grade", "stage", "ith",
    ]),
    ("Clinical + WGI + ITH", [
        "punctuated", "purity", "age", "grade", "stage", "wgi", "ith",
    ]),
]

panel_d_rows = []
for model_name, predictors in model_specs:
    result = fit_hc3(validated, "mitoxy", predictors, "punctuated")
    panel_d_rows.append({"model": model_name, **result})
panel_d = pd.DataFrame(panel_d_rows)

if not panel_d["n"].eq(panel_d["n"].iloc[0]).all():
    raise RuntimeError("Panel D models do not use one common complete-case set")
if not panel_d["p_value"].between(0, 1).all():
    raise RuntimeError("Panel D produced invalid P values")
if not (
    (panel_d["ci_low"] <= panel_d["beta"])
    & (panel_d["beta"] <= panel_d["ci_high"])
).all():
    raise RuntimeError("Panel D confidence interval audit failed")

panel_d.to_csv(OUTPUT_DIR / "panel_d_punctuated_models.csv", index=False)
display(panel_d)


,model,n,beta,se_HC3,p_value,ci_low,ci_high
0,Clinical,73,0.570307,0.250955,0.023053,0.078445,1.062169
1,Clinical + WGI,73,0.558063,0.256309,0.029458,0.055707,1.060420
2,Clinical + ITH,73,0.493283,0.234100,0.035105,0.034455,0.952112
3,Clinical + WGI + ITH,73,0.491385,0.238078,0.039021,0.024761,0.958010


## Panel E — regional dispersion and genomic heterogeneity


In [7]:
dispersion = (
    regions.groupby("patient")["mitoxy"]
    .apply(dispersion_summary)
    .unstack()
    .reset_index()
)
dispersion = dispersion.loc[dispersion["n_regions"].ge(2)].copy()
dispersion = dispersion.merge(
    covariates, on="patient", how="left", validate="one_to_one"
)

metric_names = [
    "Range", "SD", "MAD",
    "Mean pairwise |difference|",
    "Median pairwise |difference|",
]
feature_names = {"ith": "ITH", "wgi": "WGI"}
association_rows = []

for metric in metric_names:
    for feature, feature_label in feature_names.items():
        fields = [metric, feature, "n_regions", "purity"]
        complete = dispersion[fields].replace(
            [np.inf, -np.inf], np.nan
        ).dropna()
        rho_raw, p_raw = spearmanr(complete[metric], complete[feature])
        n_partial, partial_rho, p_partial = partial_spearman(
            complete, metric, feature, ["n_regions", "purity"]
        )
        association_rows.append({
            "dispersion_metric": metric,
            "genomic_feature": feature_label,
            "n": int(len(complete)),
            "rho_raw": float(rho_raw),
            "p_raw": float(p_raw),
            "n_partial": n_partial,
            "partial_rho": partial_rho,
            "p_partial": p_partial,
        })

all_associations = pd.DataFrame(association_rows)
all_associations["fdr_partial"] = multipletests(
    all_associations["p_partial"], method="fdr_bh"
)[1]

panel_e = all_associations.loc[
    all_associations["genomic_feature"].eq("ITH"),
    [
        "dispersion_metric", "n", "rho_raw", "p_raw",
        "partial_rho", "p_partial", "fdr_partial",
    ],
].copy()
panel_e["dispersion_metric"] = pd.Categorical(
    panel_e["dispersion_metric"], categories=metric_names, ordered=True
)
panel_e = panel_e.sort_values("dispersion_metric").reset_index(drop=True)
panel_e["dispersion_metric"] = panel_e["dispersion_metric"].astype(str)

if not panel_e[[
    "rho_raw", "p_raw", "partial_rho", "p_partial", "fdr_partial"
]].apply(np.isfinite).all().all():
    raise RuntimeError("Panel E produced non-finite statistics")
if not panel_e[["p_raw", "p_partial", "fdr_partial"]].apply(
    lambda column: column.between(0, 1)
).all().all():
    raise RuntimeError("Panel E produced invalid probability values")

panel_e.to_csv(OUTPUT_DIR / "panel_e_ith_robustness.csv", index=False)
all_associations.to_csv(
    OUTPUT_DIR / "panel_e_all_heterogeneity_associations.csv", index=False
)
display(panel_e)


/scratch/ar58064/sulindac_tam/envs/sulindac-tam/lib/python3.12/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/scratch/ar58064/sulindac_tam/envs/sulindac-tam/lib/python3.12/site-packages/numpy/_core/_methods.py:211: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/scratch/ar58064/sulindac_tam/envs/sulindac-tam/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3862: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/scratch/ar58064/sulindac_tam/envs/sulindac-tam/lib/python3.12/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


,dispersion_metric,n,rho_raw,p_raw,partial_rho,p_partial,fdr_partial
0,Range,60,0.090440,0.491946,-0.158271,0.227124,0.731705
1,SD,60,-0.005263,0.968167,-0.151271,0.248604,0.731705
2,MAD,60,-0.013338,0.919436,-0.028317,0.829943,0.829943
3,Mean pairwise |difference|,60,-0.028931,0.826317,-0.137494,0.294820,0.731705
4,Median pairwise |difference|,60,-0.068582,0.602597,-0.185660,0.155536,0.731705


## Processed-file handoff


In [8]:
required_outputs = {
    "panel_b_region_scores.csv": [
        "sample_id", "patient", "mitoxy", "patient_order", "patient_label",
    ],
    "panel_b_patient_summary.csv": [
        "patient", "patient_mean", "patient_min", "patient_max", "n_regions",
        "patient_order", "patient_label",
    ],
    "panel_c_variance_partition.csv": [
        "component", "variance_fraction", "variance_percent",
    ],
    "panel_d_punctuated_models.csv": [
        "model", "n", "beta", "se_HC3", "p_value", "ci_low", "ci_high",
    ],
    "panel_e_ith_robustness.csv": [
        "dispersion_metric", "n", "rho_raw", "p_raw", "partial_rho",
        "p_partial", "fdr_partial",
    ],
}

output_rows = []
for filename, columns in required_outputs.items():
    path = OUTPUT_DIR / filename
    table = pd.read_csv(path)
    missing = sorted(set(columns) - set(table.columns))
    if missing:
        raise KeyError(f"{filename} is missing {missing}")
    output_rows.append({
        "file": filename,
        "rows": len(table),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    })

processed_manifest = pd.DataFrame(output_rows)
processed_manifest.to_csv(
    OUTPUT_DIR / "figure_4_processed_manifest.csv", index=False
)
display(processed_manifest)
print("Figure 4 processed-file contract: PASS")


,file,rows,size_bytes,sha256
0,panel_b_region_scores.csv,171,25783,33cacb7f60761e7aed6de0f8dbf34ca91d2c959e00ce4b...
1,panel_b_patient_summary.csv,58,4361,47b5c6b8251d8a2badd61afc991e1f22b151007836d140...
2,panel_c_variance_partition.csv,2,151,d069dedd3b1b519831e2cd6aaa284cd090d85bf673fae8...
3,panel_d_punctuated_models.csv,4,512,c662ebca54e82530095834beb402016b60137b120fb20b...
4,panel_e_ith_robustness.csv,5,652,b9fd7071553d9347e6bab4e2ff6ba65a1dfb00a1070fa3...


Figure 4 processed-file contract: PASS
